## Dependency Relationships

This Dynamic Bayesian Network (DBN) estimates each node state at timestamp $t$ using three sources of new failure risk and one temporal persistence term:

1. routine failure probability;
2. direct flood-induced failure probability from the fragility curve;
3. failure probability caused by dependency-parent nodes;
4. persistence of failure from the previous timestamp.

For each failure node $X_i^t$, the new failure probability at timestamp $t$ is:

$$
P_\text{new}(X_i^t=1)
=
1-
(1-P_\text{routine}(X_i))
(1-P_\text{flood}(X_i^t))
(1-P_\text{parent}(X_i^t))
$$

Temporal persistence is handled separately through the DBN recovery model:

$$
P_\text{persist}(X_i^t=1)
=
P(X_i^{t-1}=1)(1-r_i(t))
$$

The final failure probability is:

$$
P(X_i^t=1)
=
1-
(1-P_\text{persist}(X_i^t))
(1-P_\text{new}(X_i^t))
$$

Equivalently:

$$
P(X_i^t=1)
=
1-
[1-P(X_i^{t-1}=1)(1-r_i(t))]
(1-P_\text{routine}(X_i))
(1-P_\text{flood}(X_i^t))
(1-P_\text{parent}(X_i^t))
$$

Here, $P_\text{flood}(X_i^t)$ comes from the category-level flood fragility curve evaluated at the flood elevation at timestamp $t$:

$$
P_\text{flood}(X_i^t)=f_{\text{category}(i)}(h_t)
$$

where $h_t$ is the hourly flood elevation above sea level.

### Root Nodes

Root nodes do not have dependency parents, so:

$$
P_\text{parent}(X_i^t)=0
$$

The root nodes in the established HealthPark network are:

- `PP_1_failed`
- `WTP_1_failed`
- `LEE_DARK_FIBER_failed`
- `LUMEN_MPLS_failed`
- `COMCAST_ENS_failed`
- `COMCAST_BUSINESS_failed`
- `TMOBILE_5G_failed`
- `roads_bridges_closure`

### Power Chain

The power delivery chain is modeled as a sequential dependency:

$$
PP_1 \rightarrow SUB_1 \rightarrow SUB_2 \rightarrow SUB_3 \rightarrow SUB_4 \rightarrow SUB_5
$$

For each downstream substation, upstream failure makes downstream service unavailable:

$$
P(SUB_1^t=1 \mid PP_1^t=1)=1
$$

$$
P(SUB_2^t=1 \mid SUB_1^t=1)=1
$$

$$
P(SUB_3^t=1 \mid SUB_2^t=1)=1
$$

$$
P(SUB_4^t=1 \mid SUB_3^t=1)=1
$$

$$
P(SUB_5^t=1 \mid SUB_4^t=1)=1
$$

In probability-propagation form:

$$
P_\text{parent}(SUB_1^t)=P(PP_1^t=1)
$$

$$
P_\text{parent}(SUB_2^t)=P(SUB_1^t=1)
$$

$$
P_\text{parent}(SUB_3^t)=P(SUB_2^t=1)
$$

$$
P_\text{parent}(SUB_4^t)=P(SUB_3^t=1)
$$

$$
P_\text{parent}(SUB_5^t)=P(SUB_4^t=1)
$$

If the upstream parent is not failed, the downstream substation can still fail from routine failure, direct flood fragility, or temporal persistence.

### Generator Activation

`Generator_activated` is an activation node, not a failure node. It is directly determined by the final upstream substation state:

$$
P(Generator\_activated^t=1 \mid SUB_5^t=1)=1
$$

$$
P(Generator\_activated^t=1 \mid SUB_5^t=0)=0
$$

Therefore:

$$
P(Generator\_activated^t=1)=P(SUB_5^t=1)
$$

Because this is an activation node, no routine failure probability, direct flood fragility probability, or temporal recovery probability is assigned to `Generator_activated`.

### Electrical Distribution

The generator is assumed to work when activated and expire after 6.3 days. The generator expiration term is:

$$
q_t =
\begin{cases}
0, & elapsed\_days \le 6.3 \\
1, & elapsed\_days > 6.3
\end{cases}
$$

The parent-caused failure probability for electrical distribution is:

$$
P_\text{parent}(ElectricalDistribution^t)
=
P(Generator\_activated^t=1)q_t
$$

Before 6.3 days, generator activation does not cause electrical distribution failure:

$$
P_\text{parent}(ElectricalDistribution^t)=0
$$

After 6.3 days, generator expiration can cause electrical distribution failure:

$$
P_\text{parent}(ElectricalDistribution^t)=P(Generator\_activated^t=1)
$$

The final probability of `ElectricalDistribution_failed` still includes routine failure, direct flood fragility from the `Generator` category, parent-caused failure from generator expiration, and temporal persistence.

### Water Distribution

Water distribution depends on the external water treatment plant and internal electrical distribution:

$$
WTP_1 \rightarrow WaterDistribution
$$

$$
ElectricalDistribution \rightarrow WaterDistribution
$$

A noisy-OR relationship is used:

$$
P_\text{parent}(WaterDistribution^t)
=
1-
(1-s_{WTP}P(WTP_1^t=1))
(1-s_E P(ElectricalDistribution^t=1))
$$

The selected dependency strengths are:

$$
s_{WTP}=1.00
$$

$$
s_E=0.50
$$

Thus, water treatment plant failure is modeled as a deterministic cause of water distribution failure, while electrical distribution failure is modeled as a strong but not perfectly deterministic cause.

### Communication Dependencies

Communication dependencies use redundancy logic. A tier communication node fails only when all redundant providers for that tier fail.

Tier 1 communication depends on Lee dark fiber, Lumen MPLS, and Comcast ENS:

$$
P_\text{parent}(Communication\_Tier1^t)
=
P(LEE\_DARK\_FIBER^t=1)
P(LUMEN\_MPLS^t=1)
P(COMCAST\_ENS^t=1)
$$

Tier 2 communication depends on Lumen MPLS and Comcast ENS:

$$
P_\text{parent}(Communication\_Tier2^t)
=
P(LUMEN\_MPLS^t=1)
P(COMCAST\_ENS^t=1)
$$

Tier 3 communication depends on Lumen MPLS, Comcast Business, and T-Mobile 5G:

$$
P_\text{parent}(Communication\_Tier3^t)
=
P(LUMEN\_MPLS^t=1)
P(COMCAST\_BUSINESS^t=1)
P(TMOBILE\_5G^t=1)
$$

This structure represents provider redundancy: a single provider failure does not necessarily cause communication failure for the tier.

### Operations Capability

Operations capability depends on transportation access:

$$
roads\_bridges\_closure \rightarrow Operations\_capability\_reduced
$$

This relationship is deterministic:

$$
P(Operations\_capability\_reduced^t=1 \mid roads\_bridges\_closure^t=1)=1
$$

Therefore:

$$
P_\text{parent}(Operations\_capability\_reduced^t)
=
P(roads\_bridges\_closure^t=1)
$$

### Hospital Tier Failure

Hospital tier failure is modeled using noisy-OR relationships. Each tier can fail because of electrical distribution failure, water distribution failure, communication failure, or reduced operations capability.

For Tier 1:

$$
P_\text{parent}(Tier1^t)
=
1-
(1-0.95P(ElectricalDistribution^t=1))
(1-0.85P(WaterDistribution^t=1))
(1-0.70P(Communication\_Tier1^t=1))
(1-0.60P(Operations^t=1))
$$

For Tier 2:

$$
P_\text{parent}(Tier2^t)
=
1-
(1-0.90P(ElectricalDistribution^t=1))
(1-0.80P(WaterDistribution^t=1))
(1-0.65P(Communication\_Tier2^t=1))
(1-0.55P(Operations^t=1))
$$

For Tier 3:

$$
P_\text{parent}(Tier3^t)
=
1-
(1-0.85P(ElectricalDistribution^t=1))
(1-0.75P(WaterDistribution^t=1))
(1-0.60P(Communication\_Tier3^t=1))
(1-0.50P(Operations^t=1))
$$

The dependency strengths reflect the assumption that electrical distribution and water distribution are the strongest drivers of hospital tier failure, while communication and operations disruptions can degrade functionality but are less deterministic.

# Temporal persistence (TBC)

Temporal persistence represents whether a node that failed at the previous timestamp remains failed or recovers before the current timestamp.

For node $X_i$:

$$
P_\text{persist}(X_i^t=1)
=
P(X_i^{t-1}=1)(1-r_i(t))
$$

where $r_i(t)$ is the time-varying recovery probability.

Recovery is assumed to depend on flood intensity and flood trend:

$$
r_i(t)=r_{base,i} \cdot A_i(h_t) \cdot R(h_t-h_{t-1})
$$

The base recovery probability is derived from mean accessible recovery time:

$$
r_{base,i}=1-e^{-1/T_i}
$$

where $T_i$ is mean recovery time in hours under accessible conditions.

The access factor is:

$$
A_i(h_t)=
\begin{cases}
0.05, & h_t \ge h_{blocked,i} \\
0.25, & h_{caution,i} \le h_t < h_{blocked,i} \\
1.00, & h_t < h_{caution,i}
\end{cases}
$$

The flood trend factor is:

$$
R(h_t-h_{t-1})=
\begin{cases}
0.25, & h_t > h_{t-1}+\epsilon \\
0.60, & |h_t-h_{t-1}| \le \epsilon \\
1.00, & h_t < h_{t-1}-\epsilon
\end{cases}
$$

`Generator_activated` has no temporal recovery model because it is recalculated directly from `SUB_5_failed` at every timestamp.